# 🐛 Bug Hunter Agent Framework — Week 8

An agentic AI system that scrapes real buggy Python code from Stack Overflow, analyzes bugs using RAG + frontier models, and produces structured dataset entries.

## Pipeline

| Agent | Role |
|-------|------|
| **CodeScannerAgent** | Scrapes Stack Overflow for self-contained Python bug questions |
| **BugAnalysisAgent** | RAG (Chroma + MiniLM) + OpenRouter frontier model → identifies bugs |
| **CodeCorrectionAgent** | Fixes the identified bugs using a frontier model |\n
| **BugStructureAgent** | Produces full JSON matching `buggy_dataset_nl.jsonl` schema |
| **BugReportAgent** | Generates markdown reports + saves entries to JSONL |

> **Requires**: `OPENROUTER_API_KEY` in your `.env` file. Get one free at [openrouter.ai](https://openrouter.ai/)

> **Disclaimer:** Please ensure you have the `buggy_dataset_nl.jsonl` file placed in *this same directory*. You can download it from this [Google Drive folder](https://drive.google.com/drive/folders/1AaLC3nnm6gJLU66R7mAYFuSQRkxC0WqV?usp=drive_link).
> The `bug_agents/` folder with all Python modules must also be present alongside this notebook.

## Phase 1 — Setup & Dependencies

In [ ]:
%pip install -q requests beautifulsoup4 chromadb sentence-transformers pydantic gradio plotly scikit-learn numpy python-dotenv openai

In [1]:
import os
import sys
import json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# Load environment variables
load_dotenv(override=True)

# Verify API key
api_key = os.getenv("OPENROUTER_API_KEY", "")
if api_key:
    print(f"✓ OpenRouter API key loaded (ends with ...{api_key[-4:]})")
else:
    print("⚠ OPENROUTER_API_KEY not found! Add it to your .env file.")

✓ OpenRouter API key loaded (ends with ...56ba)


## Phase 2 — Build RAG Vectorstore

Load the existing `buggy_dataset_nl.jsonl` (60 entries) into a Chroma vectorstore.
This gives the BugAnalysisAgent RAG context — similar bugs from the dataset to use as few-shot examples.

**Inspired by Week 5 Day 5**: native Chroma `PersistentClient` + `sentence-transformers/all-MiniLM-L6-v2`.

In [ ]:
from bug_agents.analysis_agent import BugAnalysisAgent

# Initialize and build the vectorstore
analyzer = BugAnalysisAgent(db_path="bugs_vectorstore")
analyzer.build_vectorstore("buggy_dataset_nl.jsonl")

### Visualize the Vectorstore

Let's see how the bug embeddings cluster by difficulty level (easy/medium/hard).

In [ ]:
from chromadb import PersistentClient

# Load vectorstore data
chroma = PersistentClient(path="bugs_vectorstore")
collection = chroma.get_or_create_collection("bug_examples")
result = collection.get(include=["embeddings", "documents", "metadatas"])

vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
levels = [m.get("level", "unknown") for m in metadatas]

# Color by difficulty
color_map = {"easy": "#22c55e", "medium": "#eab308", "hard": "#ef4444", "unknown": "#6b7280"}
colors = [color_map.get(level, "#6b7280") for level in levels]

print(f"Loaded {len(vectors)} bug embeddings from vectorstore")

In [ ]:
# 2D t-SNE Visualization
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(15, len(vectors) - 1))
reduced_2d = tsne_2d.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_2d[:, 0], y=reduced_2d[:, 1],
    mode="markers",
    marker=dict(size=8, color=colors, opacity=0.8),
    text=[f"Level: {level}<br>Bugs: {m.get('num_bugs', '?')}<br>{doc[:80]}..."
          for level, m, doc in zip(levels, metadatas, documents)],
    hoverinfo="text",
)])
fig.update_layout(
    title="Bug Embeddings — 2D t-SNE (colored by difficulty)",
    width=800, height=500,
    margin=dict(r=20, b=10, l=10, t=40),
)
fig.show()

In [ ]:
# 3D t-SNE Visualization
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(15, len(vectors) - 1))
reduced_3d = tsne_3d.fit_transform(vectors)

fig3d = go.Figure(data=[go.Scatter3d(
    x=reduced_3d[:, 0], y=reduced_3d[:, 1], z=reduced_3d[:, 2],
    mode="markers",
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Level: {level}<br>Bugs: {m.get('num_bugs', '?')}<br>{doc[:80]}..."
          for level, m, doc in zip(levels, metadatas, documents)],
    hoverinfo="text",
)])
fig3d.update_layout(
    title="Bug Embeddings — 3D t-SNE (colored by difficulty)",
    width=900, height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)
fig3d.show()

## Phase 3 — Run the Bug Hunter Framework

Import and run the full multi-agent pipeline:

```
CodeScannerAgent → BugAnalysisAgent → BugStructureAgent → BugReportAgent
```

In [ ]:
from bug_agents.bug_hunter_framework import BugHunterFramework

framework = BugHunterFramework(
    dataset_path="buggy_dataset_nl.jsonl",
    db_path="bugs_vectorstore",
    output_path="new_bugs.jsonl",
)

In [ ]:
# Run the pipeline! Set use_test_data=True to skip Stack Overflow API
reports = framework.run(use_test_data=False)

In [ ]:
# View the markdown report
from IPython.display import Markdown, display

display(Markdown(framework.get_report_markdown()))

In [ ]:
# View the JSON entries
print(framework.get_entries_json())

## Phase 4 — Gradio UI

Interactive interface to scan, analyze, and view bug reports.

In [ ]:
import gradio as gr


def run_hunt(use_test_data):
    """Run the bug hunting pipeline and return results."""
    fw = BugHunterFramework(
        dataset_path="buggy_dataset_nl.jsonl",
        db_path="bugs_vectorstore",
        output_path="new_bugs.jsonl",
    )
    fw.build_vectorstore()
    reports = fw.run(use_test_data=use_test_data)
    md_report = fw.get_report_markdown()
    json_entries = fw.get_entries_json()
    return md_report, json_entries


with gr.Blocks(
    title="🐛 Bug Hunter Agent Framework",
    theme=gr.themes.Soft(),
) as demo:
    gr.Markdown("# 🐛 Bug Hunter Agent Framework")
    gr.Markdown(
        "Scans Stack Overflow for buggy Python code, analyzes bugs with RAG + frontier models, "
        "and produces structured dataset entries."
    )

    with gr.Row():
        test_mode = gr.Checkbox(
            label="Use test data (skip Stack Overflow API)",
            value=False,
        )
        run_btn = gr.Button("🔍 Start Bug Hunt", variant="primary", scale=2)

    with gr.Tabs():
        with gr.Tab("📋 Bug Reports"):
            report_output = gr.Markdown("Click 'Start Bug Hunt' to begin...")

        with gr.Tab("📦 JSON Entries"):
            gr.Markdown("Raw JSONL entries for appending to `buggy_dataset_nl.jsonl`:")
            json_output = gr.Textbox(
                label="New Entries (JSONL)",
                lines=15,
                interactive=False,
            )

    run_btn.click(
        fn=run_hunt,
        inputs=[test_mode],
        outputs=[report_output, json_output],
    )

demo.launch()